# М01. Когда прогноз встречает вещество: расследование растворимости

**Авторская учебная адаптация для курса «Основы хемоинформатики и вычислительной химии».** Автор и аффилиация курса: не указаны в исходном проекте; перед публикацией заполняются владельцем. Научный источник кейса — John S. Delaney, 2004; библиографические сведения ниже. Текст не является рассказом о действиях автора статьи.

## Аннотация

Прежде чем исследовать действие вещества, его приходится приготовить в подходящей среде. Если расчёт растворимости создаёт неверное представление о доступной концентрации, дальнейший эксперимент может отвечать совсем не на задуманный вопрос. В этом разборе мы исследуем опубликованный архив измеренных и рассчитанных растворимостей: проверяем происхождение данных, переводим химический вопрос в математический, сравниваем прогнозы с наблюдениями средствами Python и matplotlib. Общая ошибка постепенно уступит место отдельной записи, для которой потребуется новое исследовательское решение. Практический результат — воспроизводимый анализ и обоснованный план следующей проверки.

**О чём исследование и где данные — откройте до первой ячейки кода:**

- **Статья:** Delaney J. S. *ESOL: Estimating Aqueous Solubility Directly from Molecular Structure*. J. Chem. Inf. Comput. Sci. 2004, 44(3), 1000–1005. [DOI 10.1021/ci034243x](https://doi.org/10.1021/ci034243x), [авторская аннотация и метаданные в PubMed](https://pubmed.ncbi.nlm.nih.gov/15154768/).
- **Используемый датасет:** [CSV DeepChem, неизменяемая версия](https://raw.githubusercontent.com/deepchem/deepchem/455d07f3e17e880a4d980b5b0365e4a02b417e14/datasets/delaney-processed.csv), [страница этой версии в GitHub](https://github.com/deepchem/deepchem/blob/455d07f3e17e880a4d980b5b0365e4a02b417e14/datasets/delaney-processed.csv). Это вторичный обработанный набор из 1128 записей.
- **Паспорт данных:** [версия, SHA256, поля, единицы, права и учебные преобразования](RESEARCH_SOURCE.md). Отдельный [словарь терминов и сокращений](GLOSSARY.md) можно держать открытым рядом с notebook.

## Пролог. До нажатия Run

На экране — таблица. В одной колонке записан результат измерения, в другой — результат расчёта. Числа стоят рядом так уверенно, будто уже договорились между собой. Остаётся провести диагональ, получить среднюю ошибку и написать: «Модель подходит».

Но лабораторное решение не живёт в среднем. Для следующего опыта понадобится конкретное вещество, конкретная среда и конкретная концентрация. Именно в этой точке гладкий график должен встретиться с содержимым пробирки.

Это **учебная ситуация принятия решения**, а не реконструкция лаборатории Delaney. Мы будем работать с подлинными архивными числами и спрашивать: **что разрешают утверждать эти данные и какую неопределённость надо устранить до следующего эксперимента?**

### Как читать этот разбор

Путь состоит из трёх частей: понять предмет и сформулировать вопрос; построить проверку и выполнить расчёт; объяснить результат и принять ограниченное решение. Перед несколькими вычислениями есть короткая остановка: сначала ответьте своими словами, затем откройте пояснение. Цель — заметить изменение собственного рассуждения. Совпадение первого предположения с ответом не оценивается.

Предпосылки: представление о моле и концентрации; умение запускать ячейки Jupyter сверху вниз. Логарифмы и обозначения разбираются здесь. Если чтение словаря и таблиц Python пока затрудняет, используйте пояснения рядом с кодом; знание RDKit не требуется. [Подготовка среды](README.md#окружение).

По завершении вы сможете: **сформулировать** вопрос о пригодности прогноза; **вычислить и истолковать** остаток, средние ошибки и кратность; **обосновать** следующую проверку, отделив наблюдение от гипотезы. Реальный перенос этих навыков проверяется в [самостоятельной лабораторной](02_independent_lab.ipynb).

## 1. Зачем химику исследовать растворимость

Растворимость отвечает на вопрос о составе насыщенного раствора определённого вещества в определённом растворителе. Она может выражаться разными величинами; в нашем архиве используется молярная концентрация в воде. Такое понимание соответствует [определению IUPAC, Международного союза теоретической и прикладной химии](https://goldbook.iupac.org/terms/view/S05740/1000).

Здесь легко пропустить первое различие: концентрация приготовленного раствора и растворимость — не синонимы. Можно приготовить ненасыщенный раствор; одно его измерение ещё не устанавливает предел растворимости. Скорость исчезновения твёрдого вещества тоже не равна равновесной растворимости. До сравнения чисел нужно понимать, какую величину определял метод.

В исследовательской работе прогноз полезен для предварительного выбора условий и кандидатов на измерение. Его ценность — помочь распределить ограниченные ресурсы. Ошибочное решение возникает, когда предварительную оценку принимают за доказанную возможность получить нужную концентрацию. Точную цену такой ошибки наш CSV не сообщает; экономию времени и реагентов в этой работе мы не рассчитываем.

**Актуальность кейса:** научиться проверять цепочку «структура → расчёт свойства → экспериментальное решение». В авторской аннотации Delaney описана простая регрессионная оценка растворимости из молекулярных свойств. **Научная новизна той работы** относится к предложенной модели; **наша учебная задача** — разобрать доказательства её использования на конкретном доступном архиве. Мы не объявляем повторный подсчёт ошибки новым научным методом. [Источник: Delaney, 2004](https://pubmed.ncbi.nlm.nih.gov/15154768/).

### Остановка P01 — сформулируйте цену решения

Закончите фразу: «Мне нужен прогноз растворимости, чтобы решить …; если я поверю ему без проверки, то могу ошибиться в …». Назовите действие и измеряемую величину, а не только желание «получить точную модель».

<details><summary>После собственной попытки: пример рассуждения</summary>

«Хочу выбрать, для какого вещества сначала проверить возможность приготовления раствора нужной концентрации. Если прогноз завышен, планируемая концентрация может оказаться недостижимой в выбранных условиях». Это мотивирует проверку; это ещё не утверждение, что какое-либо конкретное вещество нашего архива уже привело к неудачному опыту.
</details>

## 2. Между молекулой и числом есть несколько переходов

Строковая запись структуры **SMILES (Simplified Molecular Input Line Entry System)** позволяет передать молекулярный граф текстом. Как устроено это представление, разберём позже; сегодня сохраняем исходную строку для связи с записью. [Официальное описание Daylight](https://www.daylight.com/dayhtml/doc/theory/theory.smiles.html).

Вещество не помещается в компьютер целиком. Сначала появляется его представление — например, строка **SMILES** (*Simplified Molecular Input Line Entry System*, строковая запись молекулярной структуры). Затем из представления получают признаки, или **дескрипторы**: числа, описывающие отдельные свойства структуры. Модель связывает признаки с предполагаемым значением свойства. Измерение даёт отдельное наблюдение при определённых условиях. Сопоставить их — ещё одно исследовательское действие.

| Переход | Что остаётся в записи | Что нельзя молча предположить |
|---|---|---|
| Образец → структура | Имя и строковое представление | Что имя однозначно задаёт форму и состояние образца |
| Структура → признаки | Например, молекулярная масса | Что признаки описывают все условия опыта |
| Признаки → прогноз | Число, вычисленное моделью | Что оно является измерением |
| Образец → наблюдение | Измеренное значение и, в хорошем протоколе, условия | Что измерение известно без погрешности |
| Сравнение → решение | Отклонение и его интерпретация | Что маленькое среднее гарантирует каждый отдельный прогноз |

**Хемоинформатика** помогает организовать представления и признаки. **Вычислительная химия** использует модели химических систем; регрессия свойств — лишь один из подходов. **Количественная связь «структура — свойство» (QSPR, Quantitative Structure–Property Relationship)** связывает структурные признаки с физико-химическим свойством. Физическая химия нужна, чтобы понимать измеряемую величину и условия; статистика — чтобы различать общую картину и отдельные отклонения. Позже появятся квантовая химия и молекулярная механика; сегодня они не нужны для расчёта остатков.

В макрокейсе A курс перейдёт к истрадефиллину и аденозиновому рецептору A2A, в макрокейсе B — к аторвастатину и HMG-CoA-редуктазе. Эти линии описаны в [карточках проверенных источников](data/evidence_cards.json). Вопрос о растворимости не совпадает с вопросом о связывании, проникновении через гематоэнцефалический барьер или концентрации в клетке. Сейчас мы учимся распознавать именно такие переходы между утверждениями.

## 3. Что у нас действительно есть

В [аннотации оригинальной статьи](https://pubmed.ncbi.nlm.nih.gov/15154768/) говорится о 2874 измеренных растворимостях, использованных при разработке модели, и о линейной регрессии по молекулярным свойствам. У нас открыт **другой конкретный объект**: вторичный обработанный [CSV DeepChem](https://github.com/deepchem/deepchem/blob/455d07f3e17e880a4d980b5b0365e4a02b417e14/datasets/delaney-processed.csv), 1128 строк. Эти размеры нельзя взаимозаменять.

Две необходимые колонки названы дословно:

| Поле CSV | Наше обозначение | Как толковать |
|---|---|---|
| `measured log solubility in mols per litre` | $y_i$ | Архивное наблюдение в логарифмической шкале |
| `ESOL predicted log solubility in mols per litre` | $\widehat y_i$ | Уже вычисленный прогноз в той же шкале |
| `Compound ID`, `smiles` | имя, представление | Контекст записи, не доказательство условий измерения |
| Нулевой порядковый номер | `row_id` | Добавляемый нами адрес строки данной версии, не химический идентификатор |

**ESOL** означает *Estimated SOLubility*, оценённая растворимость. Мы не получаем заново коэффициенты модели и не контролируем, какие наблюдения использовали для вычисления готового столбца. Поэтому наша ошибка — характеристика сравнения архивных чисел; назвать её независимой точностью на новых молекулах нельзя.

Версия источника зафиксирована commit `455d07f3e17e880a4d980b5b0365e4a02b417e14`. SHA256 — контрольная сумма байтов файла — должна быть `8c06a76f0c6487d29ab0f903e6a7a7139f189ab3c1178f159c8be8964602f189`. Сумма подтверждает, что мы читаем тот же файл; достоверность химического измерения она не доказывает. Проверен [MIT LICENSE репозитория DeepChem](https://raw.githubusercontent.com/deepchem/deepchem/455d07f3e17e880a4d980b5b0365e4a02b417e14/LICENSE); отдельное право перепечатывать оригинальное дополнение ACS не заявляется. Получение локального кэша и ограничения описаны в [паспорте](RESEARCH_SOURCE.md).

### Остановка P02 — проверьте собственное толкование источника

Можно ли написать: «Мы воспроизвели исследование Delaney на всех 2874 молекулах»? Назовите два отдельных основания для ответа.

<details><summary>Сверьте основания</summary>

Нет. Наш вторичный CSV содержит 1128 записей, а не полный набор статьи. Кроме того, мы сравниваем готовый столбец прогнозов и наблюдения, не воспроизводим обучение и авторскую проверку. Ссылка на статью не превращает другую процедуру в воспроизведение оригинала.
</details>

## 4. Постановка задачи: какое утверждение предстоит проверить

Наш общий вопрос — **позволяет ли малая средняя ошибка доверять каждому отдельному архивному прогнозу при планировании проверки?** Слово «каждому» делает вопрос строже, чем просьба посчитать одну метрику. Чтобы ответить, нужны распределение отклонений, конкретные записи и условия, при которых сравнение имеет смысл.

Разложим его на три проверяемые задачи. Сначала убедимся, что сравниваем две величины в одной шкале и для одной строки. Затем выясним, какую часть картины описывает среднее и какие расхождения оно скрывает. Наконец, отделим обнаруженное отклонение от объяснения его причины и выберем наблюдение, которое поможет различить альтернативные причины.

**Рабочее предположение до расчёта:** даже при MAE меньше единицы могут существовать записи с существенно большим расхождением. Это не гипотеза о механизме растворения конкретного вещества; это проверяемое утверждение о предоставленной таблице. Если все отклонения выбранной части окажутся малы по заранее названному ориентиру, наше ожидание крупного локального расхождения не подтвердится в этой части архива.

### Остановка P03 — сначала прогноз, затем проверка

Запишите, ожидаете ли вы увидеть отдельные сильные расхождения и на каком наблюдении пересмотрите ответ. Сохраните первоначальную формулировку. Не исправляйте её задним числом: позднее мы вернёмся к тому, что изменилось в вашем рассуждении.

## 5. Как химический вопрос становится математическим

Обозначим молярную растворимость $S>0$, выраженную в mol/L. В нашей записи

$$
y=\log_{10}\!\left(\frac{S}{1\;\mathrm{mol\,L^{-1}}}\right),\qquad
S=(1\;\mathrm{mol\,L^{-1}})\,10^y.
$$

Логарифм берётся от безразмерного отношения. Например, $y=-3$ соответствует $10^{-3}$ mol/L. Отрицательное число здесь не означает отрицательной концентрации. Нулевую концентрацию нельзя просто записать как logS = 0: $\log 0$ не определён, а logS = 0 соответствует 1 mol/L. Если метод сообщает только предел обнаружения, это отдельная задача обработки ограниченных наблюдений.

Модель связи структуры и свойства удобно записать как $\widehat y_i=f_\theta(\phi(x_i))$: $x_i$ — представление структуры, $\phi$ — вычисление признаков, $\theta$ — параметры. Общая линейная форма — $\beta_0+\sum_j\beta_j\phi_j(x_i)$. Это **объяснение класса моделей**, не подстановка неизвестных коэффициентов исходной статьи. В этой работе $\widehat y_i$ уже дано в CSV.

Само наблюдение зависит и от условий: концептуально $y_i=g(x_i,u_i)+\eta_i$, где $u_i$ обозначает среду, форму и протокол, а $\eta_i$ — отклонение измерения от значения, которое соответствовало бы этому протоколу. Мы не задаём распределение $\eta_i$, не оцениваем его по одному числу и не восстанавливаем неизвестные $u_i$ из названия вещества. Такая запись объясняет, почему разность двух колонок ещё не устанавливает источник ошибки.

Для сравнения вводим **остаток**

$$
e_i=\widehat y_i-y_i,\quad
\mathrm{MAE}=\frac1n\sum_{i=1}^{n}|e_i|,\quad
\mathrm{RMSE}=\sqrt{\frac1n\sum_{i=1}^{n}e_i^2}.
$$

MAE (*Mean Absolute Error*) — средняя абсолютная ошибка; RMSE (*Root Mean Squared Error*) — корень средней квадратической ошибки. Положительный остаток означает завышенный прогноз растворимости. MAE показывает средний модуль, а квадрат в RMSE усиливает вклад крупных отклонений. Оба числа относятся к выбранным записям и к этой шкале.

Из определения логарифма следует система интерпретации:

$$
\frac{\widehat S_i}{S_i}=10^{e_i},\qquad
F_i=10^{|e_i|}\ge1.
$$

$F_i$ показывает, во сколько раз больше одно из двух значений концентрации относительно другого. При $e=+1$ прогноз выше в 10 раз, при $e=-1$ — ниже в 10 раз. Это не проценты и не вероятность. Для перевода mol/L в mg/L нужна молярная масса $M$ в g/mol: $c_{mg/L}=1000MS$; одинаковое logS у разных веществ не означает одинаковое mg/L.

### Остановка P04 — объясните знаком, а не формулой

Наблюдение равно −3, прогноз равен −2. В какую сторону ошибся прогноз? Какой кратности соответствует разность? Сначала ответьте без Python.

<details><summary>Проверьте ход рассуждения</summary>

Прогноз даёт $10^{-2}$, наблюдение — $10^{-3}$ mol/L. Прогноз выше в 10 раз; остаток +1. Если получить −1, перепутан порядок вычитания. Если получить отрицательную концентрацию, смешаны исходная и логарифмическая шкалы. Числа в этой остановке синтетические и служат только проверке единиц.
</details>

## 6. Начальные условия, границы и критерии проверки

Начальные условия здесь — входные записи, их происхождение и правила сравнения. Это табличная задача анализа, а не решение дифференциального уравнения: пространственной границы с заданным значением функции у неё нет. **Физические условия измерения** и **границы применимости вывода** всё равно обязательны; отсутствие граничной задачи не разрешает забыть о температуре и pH.

| Что фиксируем до вычислений | Значение или ограничение |
|---|---|
| Объект анализа | Конкретная версия CSV, 1128 строк; raw-файл неизменяем |
| Учебная часть guided | Нулевой `row_id % 4 == 0`, 282 строки |
| Часть для самостоятельного переноса | `row_id % 4 == 1`, ещё 282 строки |
| Назначение разделения | Разные учебные записи; не независимое разделение обучения и проверки модели |
| Шкала | В обеих колонках log10(S/(mol/L)); конечные числа |
| Неизвестное в CSV | pH, температура, твёрдая форма, индивидуальная погрешность |
| Что не меняем | Измеренные значения, исходные байты, правило выбора записей после просмотра ошибок |

Для обучения зададим **ориентир проверки отдельной записи**: кратность 10, то есть $\tau=\log_{10}(10)=1$. Это наш явно учебный ориентир, **не норматив качества лекарственного вещества и не критерий Delaney**. При $|e_i|>\tau$ запись нуждается в отдельном разборе по этому ориентиру. Дополнительно вычислим долю $C_\tau=n^{-1}\sum_i\mathbf1(|e_i|\le\tau)$. Она отвечает на более конкретный вопрос, чем одно среднее: какая доля архивных пар укладывается в выбранный диапазон кратностей.

Приёмка имеет три уровня. **Данные:** совпали SHA256, схема и число строк, пары не потеряны. **Расчёт:** подписаны шкалы, верно вычислен остаток, $0\le\mathrm{MAE}\le\mathrm{RMSE}$, показаны все строки выбранной части. **Исследовательский результат:** вы объяснили среднее и хвост, не объявили причину без доказательств и предложили проверку, отвечающую вопросу. Прохождение первых двух уровней не удостоверяет третий. Пригодность модели для будущего эксперимента потребует его собственного допуска, условий и независимых данных.

## 7. От вопроса к процедуре и инструментам

Предметный маршрут: **нужная концентрация → предполагаемая растворимость → сопоставимое наблюдение → отклонение → неопределённость → следующий эксперимент**. На компьютере ему соответствует другой, связанный маршрут:

```text
Зафиксировать вопрос и ориентир кратности
Проверить версию файла, столбцы и единицы
Выделить заранее определённые учебные строки
Для каждой пары сохранить ID и вычислить predicted − measured
Обобщить остатки MAE/RMSE и долей в пределах ориентира
Построить график, исследовать наибольшее отклонение без удаления строки
Проверить альтернативные объяснения и неизменность raw
Записать ограниченный вывод и план различающего наблюдения
```

Псевдокод не объясняет причину расхождения сам по себе. Он делает процедуру видимой: можно проверить, не подменили ли пары, шкалу или правило выбора. Если потерять ID, даже правильная формула способна сравнить разные вещества.

| Возможный метод | Что он решает | Почему выбираем или откладываем |
|---|---|---|
| MAE/RMSE и график пар | Описывает имеющееся несоответствие | Выбираем: данные уже содержат пары |
| Доля в пределах кратности и разбор хвоста | Проверяет локальный риск относительно явного ориентира | Выбираем: среднее не отвечает за каждую запись |
| Обучение новой регрессии | Строит иной прогноз | Откладываем: сейчас проверяем данное сравнение, а не подбираем более удобную модель |
| Новый физический эксперимент | Даёт наблюдение при записанных условиях | Планируем после анализа; в notebook его результат не возникает |

`csv` читает таблицу, `math` задаёт математические операции, matplotlib строит изображения; это достаточно для данной задачи. `hashlib` и вспомогательный `m01.py` обеспечивают воспроизводимость файла. Библиотека химической обработки RDKit понадобится в других задачах: вычислять новые структурные признаки здесь не требуется. Файлы отчёта сохраняются в личную папку; [инструкция установки](README.md#окружение).

## 8. Реализация: открыть файл, не потеряв его смысл

Первая ячейка — техническая подготовка. Она находит папку модуля, подключает инструменты и создаёт `OUT`, каталог собственных результатов. Раскрыть её можно для проверки среды; это не химический метод. Следующие ячейки уже повторяют шаги нашего алгоритма.

In [ ]:
from pathlib import Path
import sys, json, csv, math, statistics, hashlib, importlib.util, os
import matplotlib.pyplot as plt
from IPython.display import display, Image
MODULE = next(candidate for parent in [Path.cwd(), *Path.cwd().parents]
              for candidate in [parent, parent/'course_source/modules/M01']
              if (candidate/'m01.py').is_file())
if str(MODULE) not in sys.path:
    sys.path.insert(0, str(MODULE))
from m01 import META, MEASURED, PREDICTED, locate_snapshot, checksum, format_feedback, metric_invariants, create_dossier
OUT = Path(os.environ.get('M01_WORK_DIR', str(MODULE/'learner_work')))/'guided'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable)
print('RDKit required: no; installed:', importlib.util.find_spec('rdkit') is not None)
from formative import feedback


`reflections` ниже — личный журнал остановок. Можно вписать ответы, которые вы сначала сформулировали на бумаге или в отдельной Markdown-ячейке. `None` означает, что ответ ещё не записан. Журнал сохраняется вместе с результатом; преподаватель оценивает смысл, а не факт заполнения.

In [ ]:
reflections={f'P{i:02d}':None for i in range(1,13)}
print('Личный журнал подготовлен; ответы ещё не записаны.')


После собственного объяснения можно проверить один его компонент: **A** — прогноз ниже в 10 раз; **B** — выше в 10 раз; **C** — выше на 1 %. В ячейке ниже задайте `choice_p04`; обратная связь укажет, к какому определению вернуться. Она не оценивает всё объяснение.

In [ ]:
choice_p04=None  # A, B или C после собственного объяснения
print(feedback('P04',choice_p04))


### 8.1. Проверяем не только наличие файла

`locate_snapshot()` принимает только проверенную версию. `csv.DictReader` читает каждую строку как словарь: имя колонки — ключ, содержимое ячейки — значение. Пока значения CSV остаются текстом. Посмотрите на названия полей: именно они связывают запись с математическими обозначениями из раздела 5. Успешное чтение ещё не устанавливает химическую достоверность наблюдений.

In [ ]:
raw_path=locate_snapshot()
raw_sha=checksum(raw_path)
with raw_path.open(encoding='utf-8',newline='') as stream:
    reader=csv.DictReader(stream)
    columns=reader.fieldnames
    raw=list(reader)
print('SHA256:',raw_sha,'rows:',len(raw))
print('Measurement:',MEASURED)
print('Prediction:',PREDICTED)
print('Missing conditions:',META['missing_conditions'])
assert len(raw)==1128 and raw_sha==META['sha256']


### 8.2. Сохраняем адрес строки и переводим текст в числа

В следующем фрагменте `enumerate` добавляет нулевой номер, `float` переводит числовую строку в число. Короткие имена `measured` и `predicted` соответствуют двум полным именам колонок. Мы сохраняем имя и SMILES для последующего поиска источника; структура здесь не пересчитывается и не исправляется.

In [ ]:
rows=[{'row_id':i,'name':r['Compound ID'],'smiles':r['smiles'],
       'measured':float(r[MEASURED]),'predicted':float(r[PREDICTED])}
      for i,r in enumerate(raw)]
display([{k:r[k] for k in ['row_id','name','measured','predicted']} for r in rows[:3]])
assert all(math.isfinite(r['measured']) and math.isfinite(r['predicted']) for r in rows)


### 8.3. Выделяем часть и смотрим первые остатки

Операция `% 4` возвращает остаток от деления номера на четыре. Правило отделяет учебные записи для разбора от записей лабораторной. `zip` соединяет запись с её рассчитанным остатком в том же порядке. Посмотрите на знак каждого из первых пяти значений: какое предсказание выше наблюдения, какое ниже?

### Остановка P05 — объясните строку перед средним

Выберите одну показанную пару и объясните словами знак её остатка. Если объяснение и знак расходятся, проверьте порядок вычитания сейчас: среднее по модулю позже способно скрыть эту ошибку.

In [ ]:
guided=[r for r in rows if r['row_id']%4==0]
residuals=[r['predicted']-r['measured'] for r in guided]
display([{'row_id':r['row_id'],'name':r['name'],'residual_logS':e}
         for r,e in list(zip(guided,residuals))[:5]])
assert len(guided)==282


### 8.4. Чего не говорит средняя ошибка

Теперь собираем отдельные отклонения в два числа. `abs` убирает знак для MAE; квадрат и `sqrt` дают RMSE. Деление на число записей необходимо: сумма ошибок увеличивалась бы просто от добавления строк.

**Остановка P06.** В коротком синтетическом упражнении остатки равны 0, 0, 0, 0, 5. MAE равна 1. Подходит ли каждое предсказание под ориентир $|e|\le1$? Почему одно среднее не отвечает на этот вопрос?

<details><summary>После собственной попытки</summary>

Нет: одна из пяти записей имеет |e| = 5, хотя среднее равно 1. Среднее и максимальное отклонение — разные характеристики. Этот придуманный набор проверяет логику и не заменяет реальные данные нашего исследования.
</details>

In [ ]:
mae=sum(abs(e) for e in residuals)/len(residuals)
rmse=math.sqrt(sum(e*e for e in residuals)/len(residuals))
print({'n':len(residuals),'MAE_logS':mae,'RMSE_logS':rmse})
assert 0<=mae<=rmse and math.isfinite(rmse)


Вернитесь к короткому контрпримеру P06. Что означает MAE = 1? **A** — все остатки не больше 1; **B** — прогнозы верны с вероятностью 1; **C** — известен средний модуль, но отдельная запись может выходить за 1. Выберите вариант после своего объяснения.

In [ ]:
choice_p06=None  # A, B или C после собственного объяснения
print(feedback('P06',choice_p06))


### 8.5. Применяем объявленный ориентир

Вернёмся к вопросу о конкретных прогнозах. Доля в пределах кратности помогает увидеть, сколько записей удовлетворяет **нашему учебному** ориентиру. Изменив `factor_limit` после первого прогона, вы исследуете чувствительность вывода к требованию; сам исходный результат при этом сохраните. Изменение допуска — изменение вопроса, а не улучшение модели.

In [ ]:
factor_limit=10.0  # Учебный ориентир; не химический норматив
if not math.isfinite(factor_limit) or factor_limit<1:
    raise ValueError('Кратность должна быть конечной и не меньше 1')
tau=math.log10(factor_limit)
within=sum(abs(e)<=tau for e in residuals)
coverage=within/len(residuals)
flagged=[r['row_id'] for r,e in zip(guided,residuals) if abs(e)>tau]
print({'factor_limit':factor_limit,'tau_logS':tau,'within':within,
       'n':len(residuals),'fraction_within':coverage,'outside':len(flagged)})
assert within+len(flagged)==len(residuals)
assert 0<=coverage<=1


## 9. График, на котором среднее теряет последнее слово

Пока мы смотрели на MAE, записи говорили хором. Теперь каждая получит точку. По горизонтали — наблюдение, по вертикали — прогноз. На диагонали они совпадают. Выше диагонали прогноз завышен, ниже — занижен. Оси в logS, поэтому визуальное расстояние в одну единицу соответствует десятикратному различию концентраций по вертикали при фиксированном наблюдении.

**Остановка P07.** До запуска предскажите, где окажется запись с завышением растворимости в 100 раз. Нарисуйте мысленно её положение относительно диагонали и только затем смотрите график. Цветом выделяется наибольшее |e|, выбранное кодом, а не заранее вырезанная удобная точка.

In [ ]:
worst=max(guided,key=lambda r:abs(r['predicted']-r['measured']))
xs=[r['measured'] for r in guided];ys=[r['predicted'] for r in guided]
lo=min(xs+ys)-.5;hi=max(xs+ys)+.5
fig,ax=plt.subplots(figsize=(7,5))
ax.scatter(xs,ys,s=18,alpha=.6,label='282 archived records')
ax.plot([lo,hi],[lo,hi],'k--',label='predicted = measured')
ax.scatter([worst['measured']],[worst['predicted']],color='crimson',s=65)
ax.annotate(f"row {worst['row_id']}: {worst['name']}",(worst['measured'],worst['predicted']),xytext=(5,8),textcoords='offset points')
ax.set(xlabel='Measured log10(S / (mol/L))',ylabel='Predicted log10(S / (mol/L))')
ax.legend();fig.tight_layout();fig.savefig(OUT/'parity.png',dpi=140)
display(Image(filename=str(OUT/'parity.png')));plt.close(fig)


### 9.1. От точки — обратно к веществу

В нашем snapshot код находит row 852, `uric acid` — мочевую кислоту. Не спешите объяснять причину знакомым названием. Сначала прочитаем два исходных значения и переведём логарифмический остаток в кратность. Гистограмма рядом показывает, как исключительная запись соотносится с остальными отклонениями выбранной части.

In [ ]:
worst_error=worst['predicted']-worst['measured']
fold_gap=10**abs(worst_error)
display({**worst,'residual_logS':worst_error,'concentration_discrepancy_factor':fold_gap})
assert worst['row_id']==852 and math.isclose(abs(worst_error),3.389,abs_tol=1e-10)
assert fold_gap>1000
fig,ax=plt.subplots(figsize=(7,3))
ax.hist([abs(e) for e in residuals],bins=20)
ax.set(xlabel='Absolute residual, logS units',ylabel='Number of archived records')
fig.tight_layout();fig.savefig(OUT/'absolute_errors.png',dpi=140)
display(Image(filename=str(OUT/'absolute_errors.png')));plt.close(fig)


## 10. Что произошло — и чего мы пока не знаем

Для этой строки measured = −3.93, predicted = −0.541. Положительный остаток +3.389 означает завышенный прогноз; отношение соответствующих концентраций составляет примерно 2449. Числа выведены из [указанного CSV](https://github.com/deepchem/deepchem/blob/455d07f3e17e880a4d980b5b0365e4a02b417e14/datasets/delaney-processed.csv), а не получены новым измерением в нашей работе. MAE около 0.77 logS этого индивидуального разрыва не отменяет.

Можно уверенно утверждать: средняя метрика не описывает каждую запись. Нельзя пока утверждать: «Причина — конкретный механизм», «эксперимент ошибочен» или «модель никогда не годится». Нам не хватает условий и независимого наблюдения, чтобы различить эти выводы.

### Остановка P08 — устройте конкуренцию объяснений

Предложите две разные причины расхождения и по одной проверке, результаты которой будут различать эти причины. Не записывайте заранее желаемый результат проверки.

<details><summary>Опора для сравнения гипотез</summary>

Конфликт шкал проверяют по определению колонок, единицам и шагам преобразования. Ограничение модели и несопоставимые условия нельзя надёжно различить без дополнительного протокола/наблюдения. Проверка того же hash исключает смену файла, но не доказывает корректность первичного измерения. Причины могут сочетаться; наш расчёт не устанавливает их взаимоисключительность.
</details>

| Версия объяснения | Что уже можно проверить | Какого доказательства недостаёт |
|---|---|---|
| Перепутаны строки или шкалы | ID, названия, формулы, журнал импорта | При сомнениях — описание исходной обработки |
| Условия наблюдений несопоставимы | Наличие соответствующих полей | pH, температура, форма, метод и неопределённость |
| Модель недостаточно описывает эту ситуацию | Само отклонение в архиве | Независимые сопоставимые данные и проверка области применимости |

### Остановка P09 — проверьте границу знания

Если в таблице нет температуры, можно ли для удобства принять 25 °C и написать, что при ней модель ошибается в 2449 раз? Объясните, какую часть утверждения поддерживают данные, а какую мы добавили бы сами.

<details><summary>Обратная связь</summary>

Данные поддерживают разрыв между двумя архивными числами. Они не сообщают выбранную температуру. 25 °C можно явно задать как условие будущего опыта, но нельзя приписать неизвестному прошлому наблюдению.
</details>

Следующая ячейка сохраняет такую границу явно. Неизменный исходник и список неизвестных нужны следующему исследователю не меньше, чем итоговая MAE.

In [ ]:
assert checksum(raw_path)==raw_sha
audit={'source_sha256':raw_sha,'unit':META['units'],'missing_conditions':META['missing_conditions'],
       'claim':'archive comparison, not independent model validation','raw_unchanged':True}
print(audit)


Проверим P09: **A** — 25 °C можно предложить для будущего опыта, но нельзя приписать архиву; **B** — можно принять обычную температуру лаборатории как факт; **C** — без температуры вообще нельзя вычислить разность колонок. Сначала сформулируйте основание, затем выберите.

In [ ]:
choice_p09=None  # A, B или C после собственного объяснения
print(feedback('P09',choice_p09))


## 11. Решение, которое можно передать коллеге

В начале хотелось решить: «Можно ли доверять прогнозу?» Теперь вопрос стал точнее. Для предварительного планирования прогноз даёт полезную проверяемую гипотезу; сам по себе он не подтверждает возможность получить нужную концентрацию для каждого вещества. Записи за пределами выбранного ориентира сохраняем и разбираем отдельно. Для спорной записи следующий шаг — установить происхождение наблюдения, форму образца и условия, затем спланировать сопоставимое измерение.

Хороший план называет величину, условия, контроль и наблюдение, которое заставит отказаться от гипотезы. Например: сначала запросить первичный протокол и выяснить, сопоставимы ли определения растворимости; при недостаточности протокола предложить повторное измерение с указанными температурой, pH, формой и оценкой неопределённости. Конкретные значения этих условий должен обосновать проект будущего эксперимента; наш архив их не задаёт. Новый результат в этом notebook не изображается полученным.

**Остановка P10.** Вам предлагают вместо проверки протокола сразу обучить более сложную модель. Какую неопределённость она устранит, а какую оставит? Выберите следующий шаг и объясните его преимущество перед одной альтернативой.

В дальнейшей линии A вопрос может касаться центральной нервной системы (ЦНС; CNS, *Central Nervous System*) или рецептора A2A. В самостоятельной линии B — концентрации аторвастатина в клетках печени. Растворимость, связывание и тканевая экспозиция — разные измеряемые характеристики. Переносится **способ исследования**, а не численный ответ ESOL на все остальные вопросы. [Источники контекста A/B](data/evidence_cards.json).

### Остановка P10 — выберите действие по неопределённости

Какую **одну** проверку вы поставите первой и с какой разумной альтернативой её сравните? Объясните, что именно её наблюдаемый результат позволит решить. Допишите ответ в `reflections['P10']`. Следующая ячейка сохраняет вычисления и текущую версию журнала.

In [ ]:
run={'source':META['source_id'],'source_sha256':raw_sha,'subset_rule':'row_id % 4 == 0',
     'rows':len(guided),'MAE_logS':mae,'RMSE_logS':rmse,'worst_row':worst,
     'worst_residual_logS':worst_error,'fold_gap':fold_gap,'audit':audit,
     'python':sys.version,'executable':sys.executable,'experiment_performed':False}
run['instructional_criterion']={'factor_limit':factor_limit,'tau_logS':tau,'fraction_within':coverage,'outside_rows':len(flagged)}
run['reflections']=reflections
(OUT/'guided_result.json').write_text(json.dumps(run,ensure_ascii=False,indent=2),encoding='utf-8')
print(OUT/'guided_result.json')


## 12. Вернуться к первоначальному вопросу

На экране осталась та же таблица. Изменилось то, что мы разрешаем себе из неё заключить. Проверка не превратила прогноз в эксперимент; она сделала видимой границу между ними. Рядом с метрикой теперь стоят отдельная запись, неизвестные условия и план следующего наблюдения.

**Остановка P11 — объяснение без кода.** Закройте расчёт и за пять предложений расскажите коллеге: зачем работали; что было дано; какую величину сравнили; какой результат изменил решение; чего не доказали и что проверите следующим. Затем сравните ответ с первоначальным P03. Найдите одно изменившееся основание, а не только новую цифру.

В карте понятий показанная связь может выглядеть так: «остаток logS» → «задаёт кратность расхождения концентраций», потому что логарифм десятичный. Источник этой связи — определение шкалы и алгебра; ограничение — она не объясняет механизм и не описывает клинический эффект. В [лабораторной](02_independent_lab.ipynb) создайте собственное ребро с направлением, объяснением, источником и следующей проверкой.

**Возврат P12 на следующий день:** без подсказки восстановите смысл знака остатка, отличие MAE от ограничения для каждого прогноза и одну неизвестную экспериментальную переменную. Затем проверьте себя по словарю. Это задание на воспроизведение понимания, не доказательство того, что оно уже сформировано.

## Источники и материалы рядом

1. [Delaney, 2004, DOI 10.1021/ci034243x](https://doi.org/10.1021/ci034243x), [аннотация](https://pubmed.ncbi.nlm.nih.gov/15154768/) — происхождение научного вопроса и класса модели.
2. [DeepChem, закреплённый CSV](https://github.com/deepchem/deepchem/blob/455d07f3e17e880a4d980b5b0365e4a02b417e14/datasets/delaney-processed.csv) — фактически анализируемые числа. [Паспорт и права](RESEARCH_SOURCE.md).
3. [IUPAC Gold Book: solubility](https://goldbook.iupac.org/terms/view/S05740/1000) — термин растворимости.
4. [Словарь](GLOSSARY.md), [математический протокол](MATHEMATICAL_PROTOCOL.md), [инструкция среды](README.md#окружение), [задания](02_independent_lab.ipynb).